## Imports

In [ ]:
from pathlib import Path
from typing import Literal

import lightning as L
import matplotlib.pyplot as plt
import torch
import yaml
from einops import rearrange
from torch import nn
from tqdm.auto import tqdm

from ntt.core.data import NTTDataModule, NTTDataModuleConfig
from ntt.core.data.components.config import DataLoaderConfig
from ntt.core.model.tokenizer import create_tokenizer

%load_ext autoreload
%autoreload 2

## Setup

In [ ]:
# DATA_ROOT = "/path/to/data/root"
DATA_ROOT = Path.home() / "ntt" / "data"
SEED = 42
BATCH_SIZE = 8
NUM_DATALOADER_WORKERS = 4

# Set seed for random number generators in pytorch, numpy and python.random
if SEED is not None:
    L.seed_everything(SEED, workers=True)

## Tokenizer

Let's create a custom tokenizer!

For now, let's consider 100 ms of binned spike counts (with 1ms bins) from a single neuron at a time. We'll design a super simple (dumb?) model with a small encoder-decoder architecture made up of fully connected layers, with a VAE bottleneck.

### Creating a Tokenizer Spec

To get started with a new tokenizer, you'll need to create a `TokenizerSpec`. This is where you tell the system what your tokenizer needs and what it can do!

A `TokenizerSpec` should contain:

1. ***Shared parameters*** needed by the datamodule (inherited from the base `TokenizerSpec` class):

    `n_neurons` (int | Dist | None):  Number of neurons to use. If this is a Dist, the number of neurons will be sampled from this distribution. None -> full population
    
    `window_length_seconds` (float | Dist): Duration of input window in seconds
    
    `bin_size_seconds` (float | None): Temporal bin size. None -> spike times (unbinned)

2. ***Algorithm-specific parameters*** defined in subclasses

3. ***Capability metadata*** (as class attributes) describing what the tokenizer can do (corresponding to the [shared attributes reference](https://www.notion.so/2c64652ee3d48031a27fc28d403640d0?v=2c64652ee3d48039bf1c000c17cc5a00) in Notion):

    `latent_type`: Type of latent representation (continuous or discrete)

    `compress_n`: Whether the tokenizer compresses along the neuron dimension

    `compress_t`: Whether the tokenizer compresses along the time dimension

    `variable_n`: Whether the tokenizer supports variable number of neurons

    `variable_t`: Whether the tokenizer supports variable time length

    `variable_dt`: Whether the tokenizer supports variable time step (bin size)

    `input_type`: Type of input data the tokenizer expects (i.e. binned counts, rates, or raw spikes)

    `output_type`: Type of data the tokenizer reconstructs

In [ ]:
from ntt.core.model.tokenizer.base import DataType, LatentType, TokenizerSpec


class SimpleVAETokenizerSpec(TokenizerSpec, frozen=True):
    """Spec for a simple single-neuron VAE tokenizer.

    Takes 1000ms of binned spike counts (1ms bins) from one neuron
    and encodes to a continuous latent via a small FC encoder-decoder.

    Algorithm-specific parameters:
        latent_dim: Dimension of the VAE latent space
        hidden_dims: Hidden layer sizes for encoder/decoder
        beta: Weight for KL divergence term (beta-VAE)
    """

    KIND: Literal["simple_vae"] = "simple_vae"

    # Fixed architecture constraints
    n_neurons: Literal[1] = 1
    window_length_seconds: float = 1.0  # 1000 ms window
    bin_size_ms: float = 1.0  # 1 ms bins

    # Capability metadata - fixed for this architecture
    latent_type: LatentType = LatentType.CONTINUOUS
    compress_n: bool = False  # Does not compress the neuron dimension
    compress_t: bool = True  # Compresses 1000 time bins -> latent
    variable_n: bool = False  # Fixed: single neuron
    variable_t: bool = False  # Fixed: 1000ms window
    variable_dt: bool = False  # Fixed: 1ms bins
    input_type: DataType = DataType.SPIKES
    output_type: DataType = DataType.SPIKES

    # Algorithm-specific parameters
    latent_dim: int = 8
    hidden_dims: tuple[int, ...] = (64, 32)
    beta: float = 1.0  # KL weight for beta-VAE

    def get_vocab_size(self) -> int:
        """Continuous latent -> no discrete vocab."""
        raise NotImplementedError("Continuous VAE has no discrete vocabulary")

In [ ]:
# Instantiate the spec - data loading params have sensible defaults
simple_vae_spec = SimpleVAETokenizerSpec(
    # Algorithm-specific (VAE architecture) - these are configurable
    latent_dim=8,
    hidden_dims=(64, 32),
    beta=1.0,
)

print("SimpleVAETokenizerSpec created!")
print(f"  Window: {simple_vae_spec.window_length_seconds * 1000:.0f} ms")
print(f"  Bin size: {simple_vae_spec.bin_size_seconds * 1000:.0f} ms")
print(f"  Neurons: {simple_vae_spec.n_neurons} (fixed)")
print(f"  Latent dim: {simple_vae_spec.latent_dim}")
print(f"  Hidden dims: {simple_vae_spec.hidden_dims}")

### Implement Tokenizer Architecture

Now that we have a spec, we need to implement the actual tokenizer by subclassing `BaseTokenizer`.

**Key requirements:**

1. **Set `spec_class`** — A class attribute linking to your spec class. This is how the registry knows which spec goes with which tokenizer.

2. **`__init__(self, spec)`** — Receives your spec instance. Call `super().__init__(spec)` first (this stores `self.spec` and runs capability validation), then build your architecture using values from the spec.

3. **`encode(x) -> tokens`** — Takes input tensor, returns discrete token indices or continuous latents.

4. **`decode(tokens) -> reconstruction`** — Takes tokens or latents, returns reconstructed data.

5. **`forward(x) -> dict`** — Full forward pass. Must return a dict with at least `'tokens'` and `'reconstruction'`. It can also include `'loss'` for training losses such as KL divergence.

6. **`@register_tokenizer`** — Decorator that registers your tokenizer so `create_tokenizer(spec)` can find it. The registry name is inferred from the spec's `KIND` field.

**How the registry works:**

When you decorate a tokenizer with `@register_tokenizer`, it adds an entry to the global registry using the tokenizer's `spec_class`. The registry key comes from `spec_class.KIND`. Later, when you call `create_tokenizer(spec)`, the factory uses `spec.KIND` to find the matching tokenizer class and instantiate it with your spec.

In [ ]:
from ntt.core.model.tokenizer.base import BaseTokenizer, register_tokenizer


@register_tokenizer
class SimpleVAETokenizer(BaseTokenizer):
    spec_class = SimpleVAETokenizerSpec  # Links spec to implementation

    def __init__(self, spec: SimpleVAETokenizerSpec):
        super().__init__(spec)  # Stores self.spec, validates capabilities

        # Build architecture using spec parameters
        input_dim = spec.window_length_bins  # 1000

        # Encoder: input -> hidden layers -> (mu, logvar)
        encoder_layers = []
        in_features = input_dim
        for hidden_dim in spec.hidden_dims:
            encoder_layers.extend(
                [
                    nn.Linear(in_features, hidden_dim),
                    nn.ReLU(),
                ]
            )
            in_features = hidden_dim
        self.encoder = nn.Sequential(*encoder_layers)
        self.fc_mu = nn.Linear(spec.hidden_dims[-1], spec.latent_dim)
        self.fc_logvar = nn.Linear(spec.hidden_dims[-1], spec.latent_dim)

        # Decoder: latent -> hidden layers (reversed) -> output
        decoder_layers = []
        in_features = spec.latent_dim
        for hidden_dim in reversed(spec.hidden_dims):
            decoder_layers.extend(
                [
                    nn.Linear(in_features, hidden_dim),
                    nn.ReLU(),
                ]
            )
            in_features = hidden_dim
        decoder_layers.append(nn.Linear(in_features, input_dim))
        self.decoder = nn.Sequential(*decoder_layers)

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, time, 1) -> latent: (batch, latent_dim)
        h = self.encoder(x.squeeze(-1))
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        # Reparameterization trick
        z = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
        return z

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        # z: (batch, latent_dim) -> reconstruction: (batch, time, 1)
        return self.decoder(z).unsqueeze(-1)

    def forward(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        # Full forward pass with KL loss
        h = self.encoder(x.squeeze(-1))
        mu, logvar = self.fc_mu(h), self.fc_logvar(h)
        z = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
        reconstruction = self.decoder(z).unsqueeze(-1)

        # KL divergence loss
        # Sum over latent dim (dim 1), then mean over batch (dim 0)
        kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1).mean()

        return {
            "tokens": z,  # For continuous VAE, "tokens" is the latent
            "reconstruction": reconstruction,
            "loss": self.spec.beta * kl_loss,  # Use beta from spec
        }

Note that we use the `create_tokenizer` factory function to initialize the tokenizer from the spec above.

In [ ]:
# Instantiate the tokenizer from the spec using the factory
tokenizer = create_tokenizer(simple_vae_spec)

print(f"Created: {tokenizer.__class__.__name__}")
print("Architecture:")
print(f"  Encoder: {tokenizer.encoder}")
print(f"  fc_mu: {tokenizer.fc_mu}")
print(f"  fc_logvar: {tokenizer.fc_logvar}")
print(f"  Decoder: {tokenizer.decoder}")

# Test with dummy input matching expected shape: (batch, time, neurons) = (batch, 1000, 1)
dummy_input = torch.randn(BATCH_SIZE, 1000, 1)  # 4 samples, 1000 time bins, 1 neuron

with torch.no_grad():
    output = tokenizer(dummy_input)

print("\nForward pass test:")
print(f"  Input shape: {dummy_input.shape}")
print(f"  Latent (tokens) shape: {output['tokens'].shape}")
print(f"  Reconstruction shape: {output['reconstruction'].shape}")
print(f"  KL loss: {output['loss'].item():.4f}")

## DataModule

We use Lightning's DataModule to abstract the dataloading.

Right now, we are loading a single sample session from the IBL dataset to wrangle with training.

Soon, we will provide different config sets (e.g. `ntt_big`, `ntt_small` ) collating brainsets from different sources, which will act as standard training and evaluation datasets for everybody.

For this example notebook, we are using data from the `perich_miller_population_2018` brainset. 
It is data from Monkey motor cortex collected using Utah arrays in [this study](https://doi.org/10.1016/j.neuron.2018.09.030).
We chose this dataset for illustration purposes as data already went through manual curation and activity is less sparse than other brain areas.

In [ ]:
from ntt.core.data.components.config import SelectionConfig

with open("../config/data/selections/perich_miller_sample_train.yaml") as f:
    train_dataset_selection = yaml.safe_load(f)
train_dataset_selection = SelectionConfig(**train_dataset_selection)

with open("../config/data/selections/perich_miller_sample_val.yaml") as f:
    val_dataset_selection = yaml.safe_load(f)
val_dataset_selection = SelectionConfig(**val_dataset_selection)

### Instantiate DataModule

Note that the Tokenizer spec needs to be passed to the datamodule: this is because the required shape and format of the returned data matrix depends on the different tokeniser capabilities.

For example, a single neuron tokeniser will receive batches containing single neurons only. Or a tokeniser dealing with continuous rates will receive data in that format.  So, even if the underlying data used is always the same, how this data is pre-processed depends on which kind of tokenizer we are training.


In [ ]:
# Create the DataModule config
datamodule_config = NTTDataModuleConfig(
    root=DATA_ROOT,
    tokenizer_spec=simple_vae_spec,
    train_selections=[train_dataset_selection],
    val_selections=[val_dataset_selection],
    # test_selections defaults to val_selections if not provided
    dataloader=DataLoaderConfig(
        batch_size=BATCH_SIZE,
        num_workers=NUM_DATALOADER_WORKERS,
    ),
    seed=SEED,
)

print("NTTDataModuleConfig created!")
print(f"  Root: {datamodule_config.root}")
print(f"  Tokenizer spec: {type(datamodule_config.tokenizer_spec).__name__}")
print(f"  Train selections: {len(datamodule_config.train_selections)} config(s)")
print(f"  Val selections: {len(datamodule_config.val_selections)} config(s)")  # type: ignore
print(f"  Dataloader batch_size: {datamodule_config.dataloader.batch_size}")
print(f"  Seed: {datamodule_config.seed}")

In [ ]:
# Create the DataModule from config
datamodule = NTTDataModule(datamodule_config)

# Setup for training (creates datasets and samplers)
datamodule.setup("fit")

print("NTTDataModule ready!")
# print(f"  Train dataset: {len(datamodule.train_dataset)} recordings")
# print(f"  Val dataset: {len(datamodule.val_dataset)} recordings")

# Get a sample batch
train_loader = datamodule.train_dataloader()
val_loader = datamodule.val_dataloader()
sample_batch = next(iter(train_loader))
print(f"\nSample batch fields: {sample_batch._fields}")
print(f"Spike matrix shape: {sample_batch.spike_matrix.shape}")

In [ ]:
print(type(sample_batch))
print("Fields:", sample_batch._fields)
print("As dict keys:", sample_batch._asdict().keys())
print("Spike matrix shape:", sample_batch.spike_matrix.shape)
print("Mask shape:", sample_batch.pad_and_gap_mask.shape)
print("Unit ids shape:", sample_batch.unit_ids.shape)

### Inspecting the datasets

Let's briefly see also how we can examine the kind of data that is part of our datamodule, and the properties it has.
For each session in a brainset, you have access to information regarding recording date, acquisition hardware, number of units,...

In [ ]:
# Get the brainsets and session ids that are part of the train dataset

loaded_session_ids = datamodule.train_dataset.get_session_ids()
loaded_session_ids

In [ ]:
# Get all of the metadata for the first session

datamodule.train_dataset.get_recording_data(loaded_session_ids[0])

### Visualizing sample batches

Note: it is quite likely that you will sometimes get 0 spikes for a certain neuron within a 1 second window :)

Also note that each batch will have `n_neurons`, depending on what is stated in your Tokenizer spec. Across batches, time windows will not overlap (and can in fact even come from different recording sessions or datasets when we have multiple in our data config). In the case of `n_neurons > 1` though, all neurons in the batch would come from the same `window_length_seconds` long portion of the recording.

In the training dataloader, `n_neurons` are always sampled at random from the total neurons in a session. If `n_neurons` exceeds the neurons in a session, only the maximum number of neurons in the session is returned, and `pad_mask` will show what padding was applied to reach the required number of neurons.

In [ ]:
train_iterator = iter(train_loader)
sample_batch = next(train_iterator)
sample_batch._fields

In [ ]:
# Explore the batch structure
# Run multiple times to see different samples

batches_to_plot = 5
plot_data = []

for _ in range(batches_to_plot):
    sample_batch = next(train_iterator)
    plot_data.append(sample_batch.spike_matrix)

plot_data = torch.cat(plot_data, dim=0)

print("Batch fields:", sample_batch._fields)
print("\nBatch contents:")
for key, value in sample_batch._asdict().items():
    if isinstance(value, torch.Tensor):
        print(f"  {key}: {value.shape} ({value.dtype})")
    else:
        print(f"  {key}: {type(value).__name__}")

# Visualize spike data
sample_spikes = rearrange(plot_data, "batch time neurons -> time (batch neurons)")

print(f"\nSample spike matrix shape: {sample_spikes.shape}")
print(f"  Time bins: {sample_spikes.shape[0]}")
print(f"  Neurons: {sample_spikes.shape[1]}")
print(f"  Total spikes: {sample_spikes.sum().item()}")
print(f"  Sparsity: {(sample_spikes == 0).float().mean().item():.2%}")

# Create raster plot
fig, ax = plt.subplots(figsize=(12, 6))

# Get spike times and neuron indices
time_steps, neuron_indices = torch.where(sample_spikes > 0)

# Plot raster
ax.scatter(time_steps.cpu(), neuron_indices.cpu(), s=10, c="black", marker="|")
ax.set_xlabel("Time (bins)")
ax.set_ylabel("Neuron Index")
ax.set_title(
    f"Spike Raster Plot ({sample_spikes.shape[1]} neurons "
    f"from {batches_to_plot} batches)\n"
    f"{len(time_steps)} spikes across "
    f"{sample_spikes.shape[1]} neurons"
)
ax.set_xlim(0, sample_spikes.shape[0])
ax.set_ylim(-0.5, sample_spikes.shape[1] - 0.5)
plt.tight_layout()
plt.show()

# Toy train Loop

In [ ]:
# Simple training loop with metric tracking
# Note: This is just for illustration
# We will support training through pytorch lightning soon
num_epochs = 20
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

# Move tokenizer to device
tokenizer = tokenizer.to(device)

params = tokenizer.parameters()
optimizer = torch.optim.AdamW(params, lr=1e-3)

# Track metrics during training
train_losses = []
val_losses = []
train_mse = []
val_mse = []
train_mae = []
val_mae = []


# Simple reconstruction loss (MSE between input and reconstruction)
# We apply pad_and_gap_mask to only compute loss on real (non-padded) data
def compute_loss(input_spikes, output, pad_and_gap_mask):
    reconstruction = output["reconstruction"]

    mask = pad_and_gap_mask.float().to(reconstruction.device)

    squared_error = (reconstruction - input_spikes.float()) ** 2

    # Apply mask and compute mean only over real (non-padded) elements
    masked_error = squared_error * mask
    num_real_elements = mask.sum()

    # Avoid division by zero (shouldn't happen, but safety check)
    if num_real_elements > 0:
        loss = masked_error.sum() / num_real_elements
    else:
        loss = torch.tensor(0.0, device=reconstruction.device, requires_grad=True)

    return loss


def compute_metrics(input_spikes, reconstruction, pad_and_gap_mask):
    mask = pad_and_gap_mask.float().to(reconstruction.device)

    squared_error = (reconstruction - input_spikes.float()) ** 2
    abs_error = torch.abs(reconstruction - input_spikes.float())

    # Apply mask and compute mean only over real (non-padded) elements
    num_real_elements = mask.sum()

    if num_real_elements > 0:
        mse = (squared_error * mask).sum() / num_real_elements
        mae = (abs_error * mask).sum() / num_real_elements
    else:
        mse = torch.tensor(0.0, device=reconstruction.device)
        mae = torch.tensor(0.0, device=reconstruction.device)

    return mse.item(), mae.item()


for epoch in range(num_epochs):
    # Training phase
    tokenizer.train()
    train_loss = 0.0
    train_mse_epoch = 0.0
    train_mae_epoch = 0.0
    num_batches = 0

    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs} [Train]", leave=False)
    for _batch_idx, batch in enumerate(train_pbar):
        # Extract spike matrix and pad_and_gap_mask, move to device
        input_spikes = batch.spike_matrix.to(device).float()  # Shape: (batch, time, neurons)
        pad_and_gap_mask = batch.pad_and_gap_mask.to(
            device
        )  # Shape: (batch, time, neurons), True = real data

        optimizer.zero_grad()
        output = tokenizer(input_spikes)
        loss = compute_loss(input_spikes, output, pad_and_gap_mask)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        mse, mae = compute_metrics(input_spikes, output["reconstruction"], pad_and_gap_mask)
        train_mse_epoch += mse
        train_mae_epoch += mae
        num_batches += 1

        # Update progress bar with current loss
        train_pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    avg_train_loss = train_loss / num_batches if num_batches > 0 else 0.0
    avg_train_mse = train_mse_epoch / num_batches if num_batches > 0 else 0.0
    avg_train_mae = train_mae_epoch / num_batches if num_batches > 0 else 0.0
    train_losses.append(avg_train_loss)
    train_mse.append(avg_train_mse)
    train_mae.append(avg_train_mae)

    # Validation phase
    tokenizer.eval()
    val_loss = 0.0
    val_mse_epoch = 0.0
    val_mae_epoch = 0.0
    num_val_batches = 0

    val_pbar = tqdm(val_loader, desc=f"Epoch {epoch + 1}/{num_epochs} [Val]", leave=False)
    with torch.no_grad():
        for batch in val_pbar:
            input_spikes = batch.spike_matrix.to(device).float()
            pad_and_gap_mask = batch.pad_and_gap_mask.to(
                device
            )  # Shape: (batch, time, neurons), True = real data
            output = tokenizer(input_spikes)
            loss = compute_loss(input_spikes, output, pad_and_gap_mask)
            val_loss += loss.item()
            mse, mae = compute_metrics(input_spikes, output["reconstruction"], pad_and_gap_mask)
            val_mse_epoch += mse
            val_mae_epoch += mae
            num_val_batches += 1

            # Update progress bar with current loss
            val_pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    avg_val_loss = val_loss / num_val_batches if num_val_batches > 0 else 0.0
    avg_val_mse = val_mse_epoch / num_val_batches if num_val_batches > 0 else 0.0
    avg_val_mae = val_mae_epoch / num_val_batches if num_val_batches > 0 else 0.0
    val_losses.append(avg_val_loss)
    val_mse.append(avg_val_mse)
    val_mae.append(avg_val_mae)

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] - Train: {avg_train_loss:.4f}, Val : {avg_val_loss:.4f}"
    )

### Training Metrics

Let's visualize how the model improved during training.

In [ ]:
# Plot training metrics
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

epochs = range(1, num_epochs + 1)

# Loss plot
axes[0].plot(epochs, train_losses, "b-", label="Train Loss", marker="o")
axes[0].plot(epochs, val_losses, "r-", label="Val Loss", marker="s")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training and Validation Loss")
axes[0].legend()

# MSE plot
axes[1].plot(epochs, train_mse, "b-", label="Train MSE", marker="o")
axes[1].plot(epochs, val_mse, "r-", label="Val MSE", marker="s")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("MSE")
axes[1].set_title("Mean Squared Error")
axes[1].legend()

# MAE plot
axes[2].plot(epochs, train_mae, "b-", label="Train MAE", marker="o")
axes[2].plot(epochs, val_mae, "r-", label="Val MAE", marker="s")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("MAE")
axes[2].set_title("Mean Absolute Error")
axes[2].legend()

plt.tight_layout()
plt.show()

### Progressive Sequence Reconstruction

Let's reconstruct a longer sequence by piecing together multiple windows from the validation set.

In [ ]:
# Reconstruct a longer sequence by concatenating multiple windows
tokenizer.eval()
num_windows = 5  # Number of batches of consecutive windows to piece together
spike_threshold = 0.05

val_iterator = iter(val_loader)
all_original = []
all_reconstructed = []

with torch.no_grad():
    for _window_idx in range(num_windows):
        try:
            batch = next(val_iterator)
        except StopIteration:
            val_iterator = iter(val_loader)
            batch = next(val_iterator)

        input_spikes = batch.spike_matrix.to(device).float()
        output = tokenizer(input_spikes)
        reconstruction = output["reconstruction"]

        # Unroll the batch dimension - each element is a sequential window
        for sample_idx in range(input_spikes.shape[0]):
            all_original.append(input_spikes[sample_idx, :, 0].cpu())
            all_reconstructed.append(reconstruction[sample_idx, :, 0].cpu())

# Concatenate all windows
original_sequence = torch.cat(all_original, dim=0)
reconstructed_sequence = torch.cat(all_reconstructed, dim=0)

# Plot the full sequence
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Original sequence
time_bins = torch.arange(len(original_sequence))
time_steps_orig = torch.where(original_sequence > 0)[0]
axes[0].scatter(
    time_steps_orig.cpu(),
    torch.zeros_like(time_steps_orig),
    s=15,
    c="black",
    marker="|",
    alpha=0.8,
)
axes[0].set_title(
    f"Original Sequence ({len(all_original)} windows, {len(time_steps_orig)} total spikes)"
)
axes[0].set_ylabel("Neuron")
axes[0].set_ylim(-0.5, 0.5)
axes[0].grid(True, alpha=0.3)

# Reconstructed sequence (as continuous values and thresholded)
axes[1].plot(
    time_bins,
    reconstructed_sequence.cpu(),
    "b-",
    alpha=0.5,
    linewidth=0.5,
    label="Continuous reconstruction",
)
recon_thresholded = (reconstructed_sequence > spike_threshold).float()
time_steps_recon = torch.where(recon_thresholded > 0)[0]
axes[1].scatter(
    time_steps_recon.cpu(),
    torch.zeros_like(time_steps_recon),
    s=15,
    c="red",
    marker="|",
    alpha=0.8,
    label=f"Thresholded spikes ({spike_threshold})",
)
axes[1].set_title(
    f"Recon Seq ({len(all_reconstructed)} windows, {len(time_steps_recon)} thresholded spikes)"
)
axes[1].set_xlabel("Time (bins)")
axes[1].set_ylabel("Neuron / Value")
axes[1].set_ylim(-0.5, max(1.0, reconstructed_sequence.max().item() * 1.1))
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Original seq: {len(original_sequence)} bins, {len(time_steps_orig)} spikes")
print(
    f"Recon. seq: {len(reconstructed_sequence)} bins, {len(time_steps_recon)} thresholded spikes"
)